# 🔧 Debug Telemetry - Fabric Racing Game

Questo notebook verifica ogni passaggio della connessione Event Hub/Eventstream.

## 1️⃣ Configurazione

Inserisci la tua Connection String dall'Eventstream Custom Endpoint:

In [ ]:
# ===========================================
# INCOLLA QUI LA TUA CONNECTION STRING
# ===========================================
CONNECTION_STRING = "Endpoint=sb://xxx.servicebus.windows.net/;SharedAccessKeyName=xxx;SharedAccessKey=xxx;EntityPath=xxx"

print("✅ Connection string configurata")
print(f"   Lunghezza: {len(CONNECTION_STRING)} caratteri")

## 2️⃣ Parsing Connection String

Verifica che la connection string contenga tutti i componenti necessari:

In [ ]:
def parse_connection_string(conn_str):
    """Parse Event Hub connection string into components."""
    parts = {}
    for part in conn_str.split(';'):
        if '=' in part:
            key, value = part.split('=', 1)
            parts[key] = value
    return parts

# Parse e verifica
parsed = parse_connection_string(CONNECTION_STRING)

print("📋 Componenti trovati nella Connection String:")
print("=" * 50)

required_keys = ['Endpoint', 'SharedAccessKeyName', 'SharedAccessKey', 'EntityPath']
all_ok = True

for key in required_keys:
    if key in parsed:
        value = parsed[key]
        # Maschera la chiave per sicurezza
        if key == 'SharedAccessKey':
            display_value = value[:8] + '...' + value[-4:] if len(value) > 12 else '***'
        else:
            display_value = value
        print(f"✅ {key}: {display_value}")
    else:
        print(f"❌ {key}: MANCANTE!")
        all_ok = False

if all_ok:
    print("\n✅ Connection String valida!")
else:
    print("\n❌ Connection String INCOMPLETA!")
    print("   Verifica di aver copiato l'intera stringa dall'Eventstream.")

## 3️⃣ Costruzione URL Event Hub

Verifica che l'URL sia costruito correttamente:

In [ ]:
# Estrai componenti
endpoint = parsed.get('Endpoint', '')
entity_path = parsed.get('EntityPath', '')

# Costruisci URL
if endpoint.startswith('sb://'):
    # Converti sb:// in https://
    host = endpoint.replace('sb://', '').rstrip('/')
    event_hub_url = f"https://{host}/{entity_path}/messages"
    print(f"✅ URL Event Hub costruito:")
    print(f"   {event_hub_url}")
else:
    print(f"❌ Endpoint non valido: {endpoint}")
    print("   Deve iniziare con 'sb://'")

# Verifica che NON sia un endpoint Kusto
if 'kusto' in event_hub_url.lower() or 'ingest-' in event_hub_url.lower():
    print("\n⚠️ ATTENZIONE: Questo sembra un endpoint Kusto, NON un Event Hub!")
    print("   Devi usare l'Eventstream Custom Endpoint, non il KQL Database URI.")
elif 'servicebus' in event_hub_url.lower():
    print("\n✅ URL contiene 'servicebus' - sembra corretto!")

## 4️⃣ Generazione SAS Token

Genera e verifica il token SAS:

In [ ]:
import hmac
import hashlib
import base64
import time
import urllib.parse

def generate_sas_token(uri, key_name, key, expiry_hours=24):
    """Generate SAS token for Event Hub."""
    # Token valido per N ore
    expiry = int(time.time() + expiry_hours * 3600)
    
    # String to sign
    string_to_sign = urllib.parse.quote_plus(uri) + '\n' + str(expiry)
    
    # Sign with HMAC-SHA256
    key_bytes = base64.b64decode(key)
    signature = hmac.new(key_bytes, string_to_sign.encode('utf-8'), hashlib.sha256).digest()
    signature_encoded = urllib.parse.quote_plus(base64.b64encode(signature).decode('utf-8'))
    
    # Build token
    token = f"SharedAccessSignature sr={urllib.parse.quote_plus(uri)}&sig={signature_encoded}&se={expiry}&skn={key_name}"
    
    return token, expiry

# Genera token
key_name = parsed.get('SharedAccessKeyName', '')
key = parsed.get('SharedAccessKey', '')

# URI per il token (senza /messages)
token_uri = f"https://{host}/{entity_path}"

try:
    sas_token, expiry_time = generate_sas_token(token_uri, key_name, key)
    expiry_readable = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(expiry_time))
    
    print("✅ SAS Token generato con successo!")
    print(f"   Scade: {expiry_readable}")
    print(f"   Lunghezza: {len(sas_token)} caratteri")
    print(f"\n   Token (primi 80 char): {sas_token[:80]}...")
except Exception as e:
    print(f"❌ Errore generazione token: {e}")
    print("   Verifica che SharedAccessKey sia in formato Base64 valido.")

## 4B️⃣ FIX: Generazione SAS Token (versione corretta)

Il problema è nel calcolo della firma. Questa versione usa l'URI nel formato corretto:

In [ ]:
import hmac
import hashlib
import base64
import time
import urllib.parse

def generate_sas_token_v2(sb_name, eh_name, key_name, key, expiry_hours=24):
    """
    Generate SAS token using the CORRECT URI format for Event Hubs.
    
    The URI for signing MUST be: sb://<namespace>.servicebus.windows.net/<eventhub>
    NOT https:// format!
    """
    # Token valido per N ore
    expiry = int(time.time() + expiry_hours * 3600)
    
    # URI per la firma - DEVE essere in formato sb://
    # IMPORTANTE: NON usare https://, ma sb://
    uri = f"sb://{sb_name}.servicebus.windows.net/{eh_name}"
    
    # Encode URI per la firma
    uri_encoded = urllib.parse.quote(uri, safe='')
    
    # String to sign: <uri_encoded>\n<expiry>
    string_to_sign = uri_encoded + '\n' + str(expiry)
    
    print(f"DEBUG - URI per firma: {uri}")
    print(f"DEBUG - URI encoded: {uri_encoded}")
    print(f"DEBUG - String to sign: {string_to_sign[:50]}...")
    
    # Sign with HMAC-SHA256
    key_bytes = base64.b64decode(key)
    signature = hmac.new(
        key_bytes, 
        string_to_sign.encode('utf-8'), 
        hashlib.sha256
    ).digest()
    signature_b64 = base64.b64encode(signature).decode('utf-8')
    signature_encoded = urllib.parse.quote(signature_b64, safe='')
    
    # Build token
    token = f"SharedAccessSignature sr={uri_encoded}&sig={signature_encoded}&se={expiry}&skn={key_name}"
    
    return token, expiry

# Estrai namespace e eventhub name dalla connection string
# Endpoint: sb://esehdbzo7515w3pyxv3hsq.servicebus.windows.net/
# EntityPath: esehdbzo7515w3pyxv3hsq_eh
namespace = host.replace('.servicebus.windows.net', '').replace('.servicebus.windows.net/', '')
eh_name = entity_path

print(f"Namespace: {namespace}")
print(f"Event Hub: {eh_name}")
print()

# Genera nuovo token
try:
    sas_token_v2, expiry_time = generate_sas_token_v2(namespace, eh_name, key_name, key)
    expiry_readable = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(expiry_time))
    
    print()
    print("✅ SAS Token V2 generato!")
    print(f"   Scade: {expiry_readable}")
    print(f"   Token (primi 100 char): {sas_token_v2[:100]}...")
except Exception as e:
    print(f"❌ Errore: {e}")
    import traceback
    traceback.print_exc()

## 4C️⃣ Test Invio con Token V2

In [ ]:
import requests
import json
from datetime import datetime

# Evento di test
test_event = {
    "EventType": "DiagnosticTestV2",
    "Timestamp": datetime.utcnow().isoformat() + "Z",
    "Source": "debug_telemetry_notebook_v2",
    "Message": "Test with fixed SAS token",
    "TestId": int(time.time())
}

# URL per invio (questo DEVE essere https://)
send_url = f"https://{namespace}.servicebus.windows.net/{eh_name}/messages"

print("📤 Invio evento con Token V2...")
print(f"   URL: {send_url}")
print(f"   Evento: {json.dumps(test_event)}")
print()

headers = {
    "Authorization": sas_token_v2,
    "Content-Type": "application/json"
}

try:
    response = requests.post(send_url, headers=headers, json=test_event, timeout=30)
    
    print(f"📥 Risposta:")
    print(f"   Status Code: {response.status_code}")
    print(f"   Status: {response.reason}")
    
    if response.status_code == 201:
        print("\n🎉 SUCCESSO! Evento inviato!")
    elif response.status_code == 200:
        print("\n🎉 SUCCESSO! (200 OK)")
    elif response.status_code == 401:
        print(f"\n❌ Ancora 401 - {response.reason}")
        print(f"   Body: {response.text}")
        
        # Proviamo un altro formato di URI
        print("\n🔄 Provo con URI in formato diverso...")
    else:
        print(f"\n⚠️ Status: {response.status_code}")
        print(f"   Body: {response.text}")
        
except Exception as e:
    print(f"❌ Errore: {e}")

## 4D️⃣ Test con URI HTTPS (alternativo)

Se il token con sb:// non funziona, proviamo con https://:

In [ ]:
def generate_sas_token_https(namespace, eh_name, key_name, key, expiry_hours=24):
    """
    Generate SAS token using HTTPS URI format.
    """
    expiry = int(time.time() + expiry_hours * 3600)
    
    # URI in formato HTTPS (senza /messages)
    uri = f"https://{namespace}.servicebus.windows.net/{eh_name}"
    
    # Encode URI
    uri_encoded = urllib.parse.quote(uri, safe='')
    
    # String to sign
    string_to_sign = uri_encoded + '\n' + str(expiry)
    
    print(f"DEBUG HTTPS - URI: {uri}")
    print(f"DEBUG HTTPS - Encoded: {uri_encoded[:60]}...")
    
    # Sign
    key_bytes = base64.b64decode(key)
    signature = hmac.new(key_bytes, string_to_sign.encode('utf-8'), hashlib.sha256).digest()
    signature_encoded = urllib.parse.quote(base64.b64encode(signature).decode('utf-8'), safe='')
    
    token = f"SharedAccessSignature sr={uri_encoded}&sig={signature_encoded}&se={expiry}&skn={key_name}"
    return token

# Genera token con URI HTTPS
sas_token_https = generate_sas_token_https(namespace, eh_name, key_name, key)
print()
print(f"Token HTTPS (primi 100): {sas_token_https[:100]}...")
print()

# Test invio
print("📤 Test invio con Token HTTPS...")
headers_https = {
    "Authorization": sas_token_https,
    "Content-Type": "application/json"
}

test_event_https = {
    "EventType": "TestHTTPS",
    "Timestamp": datetime.utcnow().isoformat() + "Z",
    "TestId": int(time.time())
}

try:
    resp = requests.post(send_url, headers=headers_https, json=test_event_https, timeout=30)
    print(f"   Status: {resp.status_code} {resp.reason}")
    if resp.status_code in [200, 201]:
        print("🎉 SUCCESSO con token HTTPS!")
    else:
        print(f"   Body: {resp.text[:200] if resp.text else 'empty'}")
except Exception as e:
    print(f"❌ {e}")

## 4E️⃣ Test con azure-eventhub SDK (metodo ufficiale)

Se i metodi manuali non funzionano, usiamo l'SDK ufficiale Azure:

In [ ]:
# Prima installa l'SDK se non presente
%pip install azure-eventhub -q

In [ ]:
from azure.eventhub import EventHubProducerClient, EventData
import json

print("📤 Test invio con Azure Event Hub SDK...")
print(f"   Connection String: {CONNECTION_STRING[:50]}...")
print()

try:
    # Crea producer usando la connection string
    producer = EventHubProducerClient.from_connection_string(CONNECTION_STRING)
    
    with producer:
        # Crea batch di eventi
        event_data_batch = producer.create_batch()
        
        # Evento di test
        test_event_sdk = {
            "EventType": "SDKTest",
            "Timestamp": datetime.utcnow().isoformat() + "Z",
            "Source": "azure-eventhub-sdk",
            "Message": "Test using official Azure SDK",
            "TestId": int(time.time())
        }
        
        # Aggiungi al batch
        event_data_batch.add(EventData(json.dumps(test_event_sdk)))
        
        # Invia
        producer.send_batch(event_data_batch)
        
    print("🎉 SUCCESSO! Evento inviato con SDK Azure!")
    print(f"   Evento: {json.dumps(test_event_sdk, indent=2)}")
    print()
    print("✅ Se questo test funziona, il problema è nella generazione manuale del SAS token.")
    print("   Il gioco dovrà usare l'SDK Azure invece delle chiamate REST dirette.")
    
except Exception as e:
    print(f"❌ Errore SDK: {type(e).__name__}: {e}")
    print()
    print("Se anche l'SDK fallisce, il problema potrebbe essere:")
    print("  - La Connection String non è valida")
    print("  - L'Eventstream Custom Endpoint non è configurato correttamente")
    print("  - Problemi di rete/firewall")

## 5️⃣ Test Invio Evento

Prova ad inviare un evento di test all'Event Hub:

In [ ]:
import requests
import json
from datetime import datetime

# Evento di test
test_event = {
    "EventType": "DiagnosticTest",
    "Timestamp": datetime.utcnow().isoformat() + "Z",
    "Source": "debug_telemetry_notebook",
    "Message": "Test event from diagnostic notebook",
    "TestId": int(time.time())
}

print("📤 Invio evento di test...")
print(f"   URL: {event_hub_url}")
print(f"   Evento: {json.dumps(test_event, indent=2)}")
print()

# Headers
headers = {
    "Authorization": sas_token,
    "Content-Type": "application/json"
}

try:
    response = requests.post(
        event_hub_url,
        headers=headers,
        json=test_event,
        timeout=30
    )
    
    print(f"📥 Risposta:")
    print(f"   Status Code: {response.status_code}")
    print(f"   Status: {response.reason}")
    
    if response.status_code == 201:
        print("\n✅ SUCCESSO! Evento inviato correttamente!")
        print("   L'evento dovrebbe apparire nell'Eventhouse entro pochi secondi.")
    elif response.status_code == 200:
        print("\n✅ SUCCESSO! (200 OK)")
    elif response.status_code == 401:
        print("\n❌ ERRORE 401 - Non autorizzato")
        print("   Possibili cause:")
        print("   - SharedAccessKey errata")
        print("   - SharedAccessKeyName errato")
        print("   - Token scaduto")
        print("   - Permessi insufficienti sulla policy")
    elif response.status_code == 404:
        print("\n❌ ERRORE 404 - Endpoint non trovato")
        print("   Possibili cause:")
        print("   - EntityPath (Event Hub name) errato")
        print("   - L'Eventstream non esiste o non è attivo")
    elif response.status_code == 400:
        print("\n❌ ERRORE 400 - Bad Request")
        print(f"   Response body: {response.text}")
    else:
        print(f"\n⚠️ Risposta inattesa: {response.status_code}")
        print(f"   Response body: {response.text}")
        
    # Mostra headers della risposta (utili per debug)
    print("\n📋 Response Headers:")
    for key, value in response.headers.items():
        print(f"   {key}: {value}")
        
except requests.exceptions.Timeout:
    print("\n❌ TIMEOUT - La richiesta ha superato i 30 secondi")
    print("   L'endpoint potrebbe essere irraggiungibile o bloccato da firewall.")
except requests.exceptions.ConnectionError as e:
    print(f"\n❌ ERRORE DI CONNESSIONE: {e}")
    print("   Verifica la connessione di rete e che l'URL sia corretto.")
except Exception as e:
    print(f"\n❌ ERRORE GENERICO: {type(e).__name__}: {e}")

## 6️⃣ Test con cURL (alternativo)

Se il test Python fallisce, prova questo comando cURL dalla shell:

In [ ]:
# Genera comando cURL per test manuale
curl_command = f'''curl -v -X POST "{event_hub_url}" \
  -H "Authorization: {sas_token}" \
  -H "Content-Type: application/json" \
  -d '{{"EventType":"CurlTest","Timestamp":"{datetime.utcnow().isoformat()}Z"}}'
'''

print("📋 Comando cURL per test manuale:")
print("=" * 60)
print(curl_command)

## 7️⃣ Verifica Eventstream Configuration

Checklist per verificare la configurazione Eventstream:

In [ ]:
print("📋 CHECKLIST EVENTSTREAM")
print("=" * 60)
print()
print("1️⃣ CUSTOM ENDPOINT (Source)")
print("   [ ] Hai aggiunto un Custom Endpoint come SOURCE dell'Eventstream?")
print("   [ ] Hai selezionato 'SAS Key Authentication'?")
print("   [ ] Hai copiato la Connection String COMPLETA (non solo l'Event Hub name)?")
print()
print("2️⃣ EVENTSTREAM TOPOLOGY")
print("   [ ] L'Eventstream è PUBBLICATO (non solo in draft)?")
print("   [ ] C'è una linea che collega Source → Destination?")
print("   [ ] Lo status dell'Eventstream è 'Running'?")
print()
print("3️⃣ DESTINATION (Eventhouse)")
print("   [ ] Hai aggiunto l'Eventhouse come DESTINATION?")
print("   [ ] Hai selezionato il KQL Database corretto?")
print("   [ ] Hai creato o selezionato una tabella di destinazione?")
print("   [ ] Il mapping dei campi è configurato?")
print()
print("4️⃣ NETWORK / FIREWALL")
print("   [ ] Il tuo notebook può raggiungere *.servicebus.windows.net?")
print("   [ ] Non ci sono firewall aziendali che bloccano la porta 443?")

## 8️⃣ Summary

Riepilogo della diagnostica:

In [ ]:
print("="*60)
print("📊 RIEPILOGO DIAGNOSTICA")
print("="*60)
print()
print(f"Endpoint:     {host}")
print(f"Entity Path:  {entity_path}")
print(f"Key Name:     {key_name}")
print(f"Event Hub URL: {event_hub_url}")
print()
print("Se l'invio test ha avuto successo (201), il problema potrebbe essere:")
print("  - L'Eventstream non è collegato all'Eventhouse")
print("  - Il mapping dei campi non è configurato")
print("  - La tabella KQL non esiste")
print()
print("Se l'invio test ha fallito, controlla:")
print("  - La Connection String (deve venire da SAS Key Authentication)")
print("  - L'Eventstream deve essere pubblicato e running")
print("  - La rete deve permettere connessioni HTTPS a servicebus.windows.net")